# Estatística e análise de erros

Esta etapa reúne os resultados finais já congelados do **DFUC2022 interno** e do **FUSeg externo**. Ela não treina, não executa inferência, não procura threshold e não escolhe modelos.

A análise distingue dois objetivos:

1. **Comparação descritiva entre conjuntos:** Dice macro nos dois conjuntos, dando peso igual a cada unidade canônica.
2. **Contextualização das métricas usadas nos desafios:** Dice macro no DFUC2022 e Dice micro (*data-based*) no FUSeg. As partições avaliadas no estudo diferem dos testes privados dos desafios.

O resultado principal continua sendo a DR-IFT no threshold congelado. O threshold 0,5 é apenas sensibilidade e o SAM é um refinador secundário.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys

PROJECT_ROOT = Path('/content/drive/MyDrive/Projeto_TCC_Lucas')
SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
%cd {PROJECT_ROOT}

from tcc_segmentation.statistical_analysis import discover_result_directory

RAIZES = [PROJECT_ROOT / 'results/final_evaluation', PROJECT_ROOT / 'results']
INTERNAL = discover_result_directory(
    RAIZES,
    marker='test_per_image_all_arms.csv',
    required_files=[
        'test_per_image_all_arms.csv',
        'test_summary_all_arms.csv',
        'paired_comparisons.csv',
    ],
)
EXTERNAL = discover_result_directory(
    RAIZES,
    marker='external_per_unit.csv',
    required_files=['external_per_unit.csv', 'external_summary.csv', 'external_sam_comparison.csv'],
)
OUTPUT = PROJECT_ROOT / 'results/day16_analysis/DAY16_FINAL_20260930'
print('Teste interno:', INTERNAL)
print('Avaliação externa:', EXTERNAL)
print('Saída:', OUTPUT)


## 1. Testes dos cálculos

Os testes confirmam que o micro-Dice usa contagens agregadas, que o bootstrap é reproduzível, que os quintis dependem somente da área da referência e que a taxonomia não altera nenhuma amostra.


In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = str(SRC_ROOT) + os.pathsep + env.get("PYTHONPATH", "")
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_statistical_analysis.py"],
    text=True,
    capture_output=True,
    env=env,
)
print(result.stdout)
print(result.stderr)
result.check_returncode()


## 2. Gerar a consolidação

O bootstrap reamostra a unidade canônica 10.000 vezes. No FUSeg, referências múltiplas já foram resumidas dentro da unidade antes desta etapa. O gap externo menos interno é descritivo e usa reamostragem independente, pois as imagens dos datasets não formam pares.


In [ ]:
command = [
    sys.executable,
    "scripts/run_statistical_analysis.py",
    "--internal-results",
    str(INTERNAL),
    "--external-results",
    str(EXTERNAL),
    "--output-dir",
    str(OUTPUT),
    "--bootstrap-samples",
    "10000",
    "--seed",
    "20260930",
]
result = subprocess.run(command, text=True, capture_output=True, env=env)
print(result.stdout)
print(result.stderr)
result.check_returncode()


## 3. Macro e micro

- **Macro:** calcula a métrica por imagem e tira a média. É a medida padronizada para comparar DFUC e FUSeg.
- **Micro:** soma TP, FP e FN das unidades e calcula a métrica uma vez. É a agregação data-based usada para contextualizar o FUSeg.

Não usamos macro em um dataset e micro no outro para estimar a mudança de domínio.


In [ ]:
import pandas as pd
from IPython.display import display, Image as DisplayImage

summary = pd.read_csv(OUTPUT / "dataset_summary_macro_micro.csv")
common = summary[
    summary.apply(
        lambda r: (r.arm_id, r.threshold_policy)
        in {("DRIFT", "selected"), ("DRIFT", "fixed_0_5"), ("DRIFT_SAM_BPM", "selected")},
        axis=1,
    )
]
display(
    common[
        [
            "dataset",
            "arm_id",
            "threshold_policy",
            "units",
            "dice_macro",
            "dice_macro_ci95_low",
            "dice_macro_ci95_high",
            "dice_micro",
            "dice_micro_ci95_low",
            "dice_micro_ci95_high",
        ]
    ].round(6)
)
display(DisplayImage(filename=str(OUTPUT / "figures/macro_micro_comparison.png")))


## 4. Generalização externa

A tabela abaixo usa somente macro contra macro. O intervalo representa a incerteza do gap descritivo FUSeg menos DFUC. Ele mede a mudança observada entre datasets, mas não isola causalmente qual característica do domínio produziu a diferença.


In [ ]:
gap = pd.read_csv(OUTPUT / "cross_dataset_macro_comparison.csv")
display(gap.round(6))


## 5. Desempenho por tamanho da lesão

Cada dataset foi dividido em cinco grupos de tamanho relativo, sem consultar o desempenho. Q1 contém as menores lesões daquele dataset e Q5 as maiores. Essa análise mostra onde o modelo encontra mais dificuldade sem transformar o FUSeg em conjunto de ajuste.


In [ ]:
area = pd.read_csv(OUTPUT / "dice_by_relative_area_quintile.csv")
display(area.round(6))
display(DisplayImage(filename=str(OUTPUT / "figures/dice_by_area_quintile.png")))


## 6. Taxonomia descritiva

Os rótulos são gerados pelas métricas, não por julgamento clínico:

- predição vazia;
- ausência de sobreposição;
- baixo Dice com recall limitante, associado a tendência de subsegmentação;
- baixo Dice com precisão limitante, associado a tendência de sobresegmentação;
- sobreposição parcial (`0,50 ≤ Dice < 0,75`);
- alta sobreposição (`Dice ≥ 0,75`).

Os casos extremos permanecem no estudo. A lista serve apenas para revisão visual e discussão.


In [ ]:
taxonomy = pd.read_csv(OUTPUT / "error_taxonomy.csv")
display(taxonomy.round(3))
display(DisplayImage(filename=str(OUTPUT / "figures/error_taxonomy.png")))
extremes = pd.read_csv(OUTPUT / "extreme_cases_for_review.csv")
display(extremes.head(24))


## 7. Painéis finais congelados

Os painéis abaixo foram definidos sem consultar o desempenho individual. Eles complementam as métricas e ajudam a observar contorno, falsos positivos, lesões pequenas e mudança de domínio.


In [ ]:
display(DisplayImage(filename=str(INTERNAL / "fixed_test_panel_performance_blind.png")))
display(DisplayImage(filename=str(EXTERNAL / "fixed_external_panel_performance_blind.png")))


## 8. O que deve ser enviado

A pasta `results/day16_analysis/DAY16_FINAL_20260930` contém as tabelas, três figuras e o relatório de auditoria. Nenhum dataset ou checkpoint é incluído.
